# Random Forest — Complete Parameter Reference

This notebook provides a thorough walkthrough of both `RandomForestClassifier` and `RandomForestRegressor` from scikit-learn.
Every parameter is explained and demonstrated individually.

**Dataset used:**
- Classifier: Iris dataset
- Regressor: Synthetically generated dataset

---

## Table of Contents
1. Imports and Data Preparation
2. Random Forest Classifier — All Parameters
3. Random Forest Regressor — All Parameters
4. Feature Importances
5. Out-of-Bag (OOB) Score
6. Warm Start — Incremental Training
7. Full Parameter Reference Table

---
## 1. Imports and Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
np.random.seed(42)

### 1.1 Classification Dataset — Iris

In [ ]:
iris = load_iris()
X_clf = iris.data
y_clf = iris.target

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf
)

print("Iris dataset shape:", X_clf.shape)
print("Classes:", iris.target_names)
print("Training samples:", X_train_clf.shape[0])
print("Test samples:", X_test_clf.shape[0])

### 1.2 Regression Dataset — Synthetic

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=10,
    n_informative=6,
    noise=20.0,
    random_state=42
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

print("Regression dataset shape:", X_reg.shape)
print("Training samples:", X_train_reg.shape[0])
print("Test samples:", X_test_reg.shape[0])
print("Target range: [{:.2f}, {:.2f}]".format(y_reg.min(), y_reg.max()))

---
## 2. Random Forest Classifier — All Parameters

### Parameter Index

| Parameter | Default |
|---|---|
| `n_estimators` | 100 |
| `criterion` | `'gini'` |
| `max_depth` | `None` |
| `min_samples_split` | 2 |
| `min_samples_leaf` | 1 |
| `min_weight_fraction_leaf` | 0.0 |
| `max_features` | `'sqrt'` |
| `max_leaf_nodes` | `None` |
| `min_impurity_decrease` | 0.0 |
| `bootstrap` | `True` |
| `oob_score` | `False` |
| `n_jobs` | `None` |
| `random_state` | `None` |
| `verbose` | 0 |
| `warm_start` | `False` |
| `class_weight` | `None` |
| `ccp_alpha` | 0.0 |
| `max_samples` | `None` |

### 2.1 `n_estimators` — Number of Trees

Controls how many decision trees are built in the forest.
Higher values generally improve accuracy but increase computation time.
The forest aggregates predictions from all trees by majority vote.

In [ ]:
estimator_values = [10, 50, 100, 200, 300]
scores = []

for n in estimator_values:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    scores.append(accuracy_score(y_test_clf, model.predict(X_test_clf)))

plt.figure(figsize=(8, 4))
plt.plot(estimator_values, scores, marker='o', color='steelblue')
plt.title('Accuracy vs n_estimators')
plt.xlabel('n_estimators')
plt.ylabel('Accuracy')
plt.grid(True)
plt.tight_layout()
plt.show()

for n, s in zip(estimator_values, scores):
    print("n_estimators = {:>4d}  |  Accuracy = {:.4f}".format(n, s))

### 2.2 `criterion` — Splitting Quality Measure

Determines the function used to measure the quality of a split at each node.

- `'gini'`: Gini impurity — measures the probability of incorrect classification.
- `'entropy'`: Information gain — based on Shannon entropy.
- `'log_loss'`: Binary cross-entropy (same as log-loss).

Gini and entropy typically produce similar results; entropy can be slightly slower.

In [ ]:
for criterion in ['gini', 'entropy', 'log_loss']:
    model = RandomForestClassifier(n_estimators=100, criterion=criterion, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("criterion = {:<12s} | Accuracy = {:.4f}".format(criterion, acc))

### 2.3 `max_depth` — Maximum Tree Depth

Sets the maximum depth each decision tree can reach.
- `None` (default): Nodes are expanded until all leaves are pure or contain fewer than `min_samples_split` samples.
- Limiting depth reduces overfitting but may increase bias.

In [ ]:
depth_values = [None, 1, 2, 3, 5, 10]

for depth in depth_values:
    model = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("max_depth = {:<5s} | Accuracy = {:.4f}".format(str(depth), acc))

### 2.4 `min_samples_split` — Minimum Samples to Split a Node

The minimum number of samples required to split an internal node.
- If an integer, that exact number is used.
- If a float, it is treated as a fraction of total training samples.

Larger values prevent the model from learning overly specific patterns (regularisation).

In [ ]:
for mss in [2, 5, 10, 20, 0.1, 0.2]:
    model = RandomForestClassifier(n_estimators=100, min_samples_split=mss, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("min_samples_split = {:<5} | Accuracy = {:.4f}".format(str(mss), acc))

### 2.5 `min_samples_leaf` — Minimum Samples at a Leaf Node

The minimum number of samples required to be present at a leaf node.
- Ensures every leaf represents at least this many training samples.
- Acts as smoothing in regression and helps prevent overfitting in classification.

In [ ]:
for msl in [1, 2, 5, 10, 0.05, 0.1]:
    model = RandomForestClassifier(n_estimators=100, min_samples_leaf=msl, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("min_samples_leaf = {:<5} | Accuracy = {:.4f}".format(str(msl), acc))

### 2.6 `min_weight_fraction_leaf` — Minimum Weighted Fraction at Leaf

The minimum weighted fraction of the sum total of weights (of all input samples) required to be at a leaf node.
- Relevant only when `sample_weight` is passed to `fit()`.
- Default `0.0` imposes no constraint.

In [ ]:
for mwfl in [0.0, 0.01, 0.05, 0.1]:
    model = RandomForestClassifier(
        n_estimators=100, min_weight_fraction_leaf=mwfl, random_state=42
    )
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("min_weight_fraction_leaf = {:.2f} | Accuracy = {:.4f}".format(mwfl, acc))

### 2.7 `max_features` — Features Considered at Each Split

Controls the number of features to consider when looking for the best split at each node.
This is the key source of randomness that decorrelates the trees.

- `'sqrt'` (default for classifier): sqrt(n_features)
- `'log2'`: log2(n_features)
- `None` or `1.0`: all features
- Integer: exact number of features
- Float in (0, 1]: fraction of features

In [ ]:
for mf in ['sqrt', 'log2', None, 1, 2, 3, 0.5]:
    model = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("max_features = {:<6} | Accuracy = {:.4f}".format(str(mf), acc))

### 2.8 `max_leaf_nodes` — Maximum Number of Leaf Nodes

Grows trees in best-first fashion up to this many leaf nodes.
- `None` (default): unlimited leaf nodes.
- Setting this constrains tree complexity independent of depth.

In [ ]:
for mln in [None, 5, 10, 20, 50]:
    model = RandomForestClassifier(n_estimators=100, max_leaf_nodes=mln, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("max_leaf_nodes = {:<5} | Accuracy = {:.4f}".format(str(mln), acc))

### 2.9 `min_impurity_decrease` — Minimum Impurity Gain to Split

A node is split only if the impurity decrease is greater than or equal to this threshold.
- Helps prune splits that offer negligible gain.
- Default `0.0` means any improvement triggers a split.

In [ ]:
for mid in [0.0, 0.001, 0.01, 0.05, 0.1]:
    model = RandomForestClassifier(
        n_estimators=100, min_impurity_decrease=mid, random_state=42
    )
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("min_impurity_decrease = {:.3f} | Accuracy = {:.4f}".format(mid, acc))

### 2.10 `bootstrap` — Bootstrap Sampling

Determines whether each tree is trained on a bootstrap sample (sampling with replacement).
- `True` (default): each tree sees a random subset of training samples (approx. 63.2% unique).
- `False`: every tree trains on the full dataset (reduces variance diversity; may overfit).
- When `bootstrap=True`, the unused samples (out-of-bag samples) can be used for internal validation.

In [ ]:
for bs in [True, False]:
    model = RandomForestClassifier(n_estimators=100, bootstrap=bs, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("bootstrap = {:<6} | Accuracy = {:.4f}".format(str(bs), acc))

### 2.11 `oob_score` — Out-of-Bag Evaluation

When `bootstrap=True`, approximately 36.8% of samples are not used to train each tree.
These are called out-of-bag (OOB) samples and can serve as an internal validation set.
- `oob_score=True` computes the OOB score after training.
- Accessible via `model.oob_score_`.
- Provides a free cross-validation estimate without additional computation.

In [ ]:
model_oob = RandomForestClassifier(
    n_estimators=100, bootstrap=True, oob_score=True, random_state=42
)
model_oob.fit(X_train_clf, y_train_clf)

test_acc = accuracy_score(y_test_clf, model_oob.predict(X_test_clf))
print("OOB Score (internal estimate): {:.4f}".format(model_oob.oob_score_))
print("Test Set Accuracy:             {:.4f}".format(test_acc))

### 2.12 `n_jobs` — Parallelism

Number of CPU cores to use when fitting and predicting.
- `None` (default): uses 1 core.
- `-1`: uses all available cores.
- Positive integer: uses that many cores.

Setting `n_jobs=-1` can significantly speed up training on large datasets.

In [ ]:
import time

for jobs in [1, -1]:
    start = time.time()
    model = RandomForestClassifier(n_estimators=200, n_jobs=jobs, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    elapsed = time.time() - start
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("n_jobs = {:<3} | Accuracy = {:.4f} | Time = {:.4f}s".format(jobs, acc, elapsed))

### 2.13 `random_state` — Reproducibility Seed

Seeds the random number generator that controls:
- Bootstrap sampling
- Feature subsampling at each split

Set to an integer for fully reproducible results.
`None` uses the global NumPy random state.

In [ ]:
for rs in [None, 0, 42, 99]:
    model = RandomForestClassifier(n_estimators=100, random_state=rs)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("random_state = {:<5} | Accuracy = {:.4f}".format(str(rs), acc))

### 2.14 `verbose` — Verbosity Level

Controls the amount of output printed during fitting and predicting.
- `0` (default): silent.
- `1`: prints progress for each tree.
- `>1`: prints even more detailed information.

Useful for monitoring long training runs.

In [ ]:
model_verbose = RandomForestClassifier(n_estimators=3, verbose=1, random_state=42)
model_verbose.fit(X_train_clf, y_train_clf)
print("\nVerbose output shown above during fit.")

### 2.15 `class_weight` — Handling Class Imbalance

Assigns weights to classes to handle imbalanced datasets.
- `None` (default): all classes have equal weight.
- `'balanced'`: weights are inversely proportional to class frequencies.
- `'balanced_subsample'`: like `'balanced'` but computed on each bootstrap sample.
- Dictionary `{class: weight}`: manually specified weights.

Using balanced weights penalises misclassifying minority classes more heavily.

In [ ]:
from sklearn.datasets import make_classification

X_imb, y_imb = make_classification(
    n_samples=500, n_features=10, weights=[0.9, 0.1],
    n_informative=5, random_state=42
)
X_tr_imb, X_te_imb, y_tr_imb, y_te_imb = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42
)

for cw in [None, 'balanced', 'balanced_subsample', {0: 1, 1: 5}]:
    model = RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42)
    model.fit(X_tr_imb, y_tr_imb)
    acc = accuracy_score(y_te_imb, model.predict(X_te_imb))
    print("class_weight = {:<25} | Accuracy = {:.4f}".format(str(cw), acc))

### 2.16 `ccp_alpha` — Cost-Complexity Pruning

Minimal cost-complexity pruning parameter.
- Subtrees with a cost-complexity smaller than `ccp_alpha` are pruned.
- `0.0` (default): no pruning.
- Larger values lead to more pruning and simpler trees.
- Introduced in scikit-learn 0.22 as a principled post-pruning approach.

In [ ]:
for alpha in [0.0, 0.001, 0.005, 0.01, 0.05]:
    model = RandomForestClassifier(n_estimators=100, ccp_alpha=alpha, random_state=42)
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("ccp_alpha = {:.3f} | Accuracy = {:.4f}".format(alpha, acc))

### 2.17 `max_samples` — Bootstrap Sample Size

When `bootstrap=True`, controls how many samples are drawn for each tree.
- `None` (default): draws `n_samples` samples (full training set size with replacement).
- Integer: draws that many samples.
- Float in (0, 1]: draws that fraction of total training samples.

Reducing `max_samples` introduces more diversity between trees, potentially improving generalisation on large datasets.

In [ ]:
for ms in [None, 0.5, 0.75, 50, 80]:
    model = RandomForestClassifier(
        n_estimators=100, bootstrap=True, max_samples=ms, random_state=42
    )
    model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, model.predict(X_test_clf))
    print("max_samples = {:<5} | Accuracy = {:.4f}".format(str(ms), acc))

### 2.18 Full Classifier Evaluation

Training a well-configured classifier and evaluating it fully.

In [ ]:
clf = RandomForestClassifier(
    n_estimators=200,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='sqrt',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None
)

clf.fit(X_train_clf, y_train_clf)
y_pred_clf = clf.predict(X_test_clf)

print("Test Accuracy : {:.4f}".format(accuracy_score(y_test_clf, y_pred_clf)))
print("OOB Score     : {:.4f}".format(clf.oob_score_))
print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_clf, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test_clf, y_pred_clf)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(iris.target_names)
ax.set_yticklabels(iris.target_names)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix — Random Forest Classifier')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Random Forest Regressor — All Parameters

The regressor shares most parameters with the classifier.
Key differences are in `criterion` and the absence of `class_weight`.

### Parameter Index

| Parameter | Default |
|---|---|
| `n_estimators` | 100 |
| `criterion` | `'squared_error'` |
| `max_depth` | `None` |
| `min_samples_split` | 2 |
| `min_samples_leaf` | 1 |
| `min_weight_fraction_leaf` | 0.0 |
| `max_features` | `1.0` |
| `max_leaf_nodes` | `None` |
| `min_impurity_decrease` | 0.0 |
| `bootstrap` | `True` |
| `oob_score` | `False` |
| `n_jobs` | `None` |
| `random_state` | `None` |
| `verbose` | 0 |
| `warm_start` | `False` |
| `ccp_alpha` | 0.0 |
| `max_samples` | `None` |

### 3.1 `criterion` — Regression Splitting Criteria

Measures the quality of a split for regression tasks.

- `'squared_error'` (default): mean squared error; minimises variance at leaf nodes.
- `'absolute_error'`: mean absolute error; more robust to outliers but slower.
- `'friedman_mse'`: Friedman's MSE with an improvement score weighting; often better in practice.
- `'poisson'`: uses Poisson deviance; suited for count data (non-negative targets).

In [ ]:
for criterion in ['squared_error', 'absolute_error', 'friedman_mse']:
    model = RandomForestRegressor(n_estimators=100, criterion=criterion, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    y_pred = model.predict(X_test_reg)
    r2 = r2_score(y_test_reg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    print("criterion = {:<15s} | R2 = {:.4f} | RMSE = {:.4f}".format(criterion, r2, rmse))

### 3.2 `max_features` — Default Differs from Classifier

For regression, the default is `1.0` (all features), unlike classification which defaults to `'sqrt'`.
Common practice is to use `'sqrt'` or `'log2'` for regression as well to increase tree diversity.

In [ ]:
for mf in [1.0, 'sqrt', 'log2', 0.5, 5]:
    model = RandomForestRegressor(n_estimators=100, max_features=mf, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    r2 = r2_score(y_test_reg, model.predict(X_test_reg))
    print("max_features = {:<6} | R2 = {:.4f}".format(str(mf), r2))

### 3.3 `max_depth`, `min_samples_split`, `min_samples_leaf` — Regressor

In [ ]:
print("--- max_depth ---")
for depth in [None, 3, 5, 10, 20]:
    model = RandomForestRegressor(n_estimators=100, max_depth=depth, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    r2 = r2_score(y_test_reg, model.predict(X_test_reg))
    print("  max_depth = {:<5} | R2 = {:.4f}".format(str(depth), r2))

print("\n--- min_samples_split ---")
for mss in [2, 5, 10, 20]:
    model = RandomForestRegressor(n_estimators=100, min_samples_split=mss, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    r2 = r2_score(y_test_reg, model.predict(X_test_reg))
    print("  min_samples_split = {:<5} | R2 = {:.4f}".format(mss, r2))

print("\n--- min_samples_leaf ---")
for msl in [1, 2, 5, 10]:
    model = RandomForestRegressor(n_estimators=100, min_samples_leaf=msl, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    r2 = r2_score(y_test_reg, model.predict(X_test_reg))
    print("  min_samples_leaf = {:<5} | R2 = {:.4f}".format(msl, r2))

### 3.4 `bootstrap`, `oob_score`, `max_samples` — Regressor

In [ ]:
model_oob_reg = RandomForestRegressor(
    n_estimators=200, bootstrap=True, oob_score=True, random_state=42
)
model_oob_reg.fit(X_train_reg, y_train_reg)

r2_test = r2_score(y_test_reg, model_oob_reg.predict(X_test_reg))
print("OOB R2 Score (internal): {:.4f}".format(model_oob_reg.oob_score_))
print("Test Set R2 Score:       {:.4f}".format(r2_test))

### 3.5 `ccp_alpha` — Pruning in Regressor

In [ ]:
for alpha in [0.0, 0.01, 0.05, 0.1, 0.5]:
    model = RandomForestRegressor(n_estimators=100, ccp_alpha=alpha, random_state=42)
    model.fit(X_train_reg, y_train_reg)
    r2 = r2_score(y_test_reg, model.predict(X_test_reg))
    print("ccp_alpha = {:.2f} | R2 = {:.4f}".format(alpha, r2))

### 3.6 Full Regressor Evaluation

In [ ]:
reg = RandomForestRegressor(
    n_estimators=200,
    criterion='squared_error',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=1.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
    verbose=0,
    warm_start=False,
    ccp_alpha=0.0,
    max_samples=None
)

reg.fit(X_train_reg, y_train_reg)
y_pred_reg = reg.predict(X_test_reg)

print("R2 Score  : {:.4f}".format(r2_score(y_test_reg, y_pred_reg)))
print("MAE       : {:.4f}".format(mean_absolute_error(y_test_reg, y_pred_reg)))
print("RMSE      : {:.4f}".format(np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))))
print("OOB Score : {:.4f}".format(reg.oob_score_))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5, color='steelblue', edgecolors='none')
min_val = min(y_test_reg.min(), y_pred_reg.min())
max_val = max(y_test_reg.max(), y_pred_reg.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linewidth=1.5, label='Perfect Fit')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted — Random Forest Regressor')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Feature Importances

Random Forest computes feature importances based on the mean decrease in impurity across all trees.
Accessible via `model.feature_importances_`.
Higher value means the feature contributes more to the prediction.

In [ ]:
importances_clf = clf.feature_importances_
feature_names_clf = iris.feature_names
sorted_idx_clf = np.argsort(importances_clf)[::-1]

plt.figure(figsize=(7, 4))
plt.bar(range(len(importances_clf)), importances_clf[sorted_idx_clf], color='steelblue')
plt.xticks(range(len(importances_clf)), [feature_names_clf[i] for i in sorted_idx_clf], rotation=20, ha='right')
plt.title('Feature Importances — Classifier (Iris)')
plt.ylabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print("Feature Importances (Classifier):")
for i in sorted_idx_clf:
    print("  {:<30s}: {:.4f}".format(feature_names_clf[i], importances_clf[i]))

In [ ]:
importances_reg = reg.feature_importances_
feature_names_reg = ["Feature {}".format(i) for i in range(X_reg.shape[1])]
sorted_idx_reg = np.argsort(importances_reg)[::-1]

plt.figure(figsize=(9, 4))
plt.bar(range(len(importances_reg)), importances_reg[sorted_idx_reg], color='darkorange')
plt.xticks(range(len(importances_reg)), [feature_names_reg[i] for i in sorted_idx_reg])
plt.title('Feature Importances — Regressor (Synthetic)')
plt.ylabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print("Feature Importances (Regressor):")
for i in sorted_idx_reg:
    print("  {:<15s}: {:.4f}".format(feature_names_reg[i], importances_reg[i]))

---
## 5. Out-of-Bag (OOB) Score — Deeper Look

The OOB score tracks model generalisation without a held-out validation set.
Comparing OOB score to test score helps detect overfitting.

In [ ]:
n_est_range = [10, 20, 50, 100, 150, 200]
oob_scores = []
test_scores = []

for n in n_est_range:
    m = RandomForestClassifier(n_estimators=n, bootstrap=True, oob_score=True, random_state=42)
    m.fit(X_train_clf, y_train_clf)
    oob_scores.append(m.oob_score_)
    test_scores.append(accuracy_score(y_test_clf, m.predict(X_test_clf)))

plt.figure(figsize=(8, 4))
plt.plot(n_est_range, oob_scores, marker='s', label='OOB Score', color='tomato')
plt.plot(n_est_range, test_scores, marker='o', label='Test Score', color='steelblue')
plt.xlabel('n_estimators')
plt.ylabel('Score')
plt.title('OOB Score vs Test Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 6. `warm_start` — Incremental Tree Addition

When `warm_start=True`, the existing fitted trees are retained and new trees are added on subsequent calls to `fit()`.
This is useful for gradually increasing the forest size without retraining from scratch.
- `False` (default): each `fit()` call discards previous trees and retrains from scratch.

In [ ]:
warm_model = RandomForestClassifier(
    n_estimators=10, warm_start=True, random_state=42
)

increments = [10, 30, 60, 100, 150]
warm_scores = []

for n in increments:
    warm_model.n_estimators = n
    warm_model.fit(X_train_clf, y_train_clf)
    acc = accuracy_score(y_test_clf, warm_model.predict(X_test_clf))
    warm_scores.append(acc)
    print("n_estimators = {:>4d} | Accuracy = {:.4f}".format(n, acc))

plt.figure(figsize=(7, 4))
plt.plot(increments, warm_scores, marker='o', color='seagreen')
plt.xlabel('Total n_estimators')
plt.ylabel('Accuracy')
plt.title('warm_start: Incrementally Adding Trees')
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 7. Full Parameter Reference Table

Summary of all parameters, their defaults, valid values, and what they control.

In [ ]:
param_table = pd.DataFrame({
    'Parameter': [
        'n_estimators', 'criterion', 'max_depth', 'min_samples_split',
        'min_samples_leaf', 'min_weight_fraction_leaf', 'max_features',
        'max_leaf_nodes', 'min_impurity_decrease', 'bootstrap',
        'oob_score', 'n_jobs', 'random_state', 'verbose',
        'warm_start', 'class_weight', 'ccp_alpha', 'max_samples'
    ],
    'Classifier Default': [
        100, 'gini', 'None', 2, 1, 0.0, 'sqrt',
        'None', 0.0, True, False, 'None', 'None', 0,
        False, 'None', 0.0, 'None'
    ],
    'Regressor Default': [
        100, 'squared_error', 'None', 2, 1, 0.0, '1.0',
        'None', 0.0, True, False, 'None', 'None', 0,
        False, 'N/A', 0.0, 'None'
    ],
    'Description': [
        'Number of trees in the forest',
        'Split quality function (gini/entropy/log_loss | squared_error/absolute_error/friedman_mse/poisson)',
        'Maximum depth of each tree (None = unlimited)',
        'Min samples required to split a node (int or float)',
        'Min samples required at a leaf node (int or float)',
        'Min weighted fraction at leaf when sample_weight is used',
        'Number of features to consider at each split (sqrt/log2/None/int/float)',
        'Max number of leaf nodes (None = unlimited)',
        'Min impurity decrease needed for a split to occur',
        'Whether to use bootstrap sampling per tree',
        'Whether to use OOB samples to estimate generalisation score',
        'Number of parallel jobs (-1 uses all cores)',
        'Random seed for reproducibility',
        'Verbosity level during fit/predict',
        'Reuse previous fit; add more trees incrementally',
        'Class weights for imbalanced data (balanced/balanced_subsample/dict) — classifier only',
        'Cost-complexity pruning threshold (0.0 = no pruning)',
        'Size of bootstrap sample per tree (None/int/float)'
    ]
})

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
print(param_table.to_string(index=False))

---
## 8. Cross-Validation Summary

In [ ]:
clf_cv = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
cv_scores_clf = cross_val_score(clf_cv, X_clf, y_clf, cv=5, scoring='accuracy')

reg_cv = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
cv_scores_reg = cross_val_score(reg_cv, X_reg, y_reg, cv=5, scoring='r2')

print("Classifier 5-Fold Cross-Validation (Accuracy):")
print("  Scores : {}".format(np.round(cv_scores_clf, 4)))
print("  Mean   : {:.4f}".format(cv_scores_clf.mean()))
print("  Std    : {:.4f}".format(cv_scores_clf.std()))

print("\nRegressor 5-Fold Cross-Validation (R2):")
print("  Scores : {}".format(np.round(cv_scores_reg, 4)))
print("  Mean   : {:.4f}".format(cv_scores_reg.mean()))
print("  Std    : {:.4f}".format(cv_scores_reg.std()))